In [0]:
import numpy as np
import pandas as pd
import mlflow
import dowhy
from dowhy import CausalModel
import warnings
warnings.filterwarnings('ignore')

gold_df = spark.table("causal_project.gold_household").toPandas()

# Same encoding as modeling notebook
gold_encoded = gold_df.copy()
gold_encoded['age_group'] = gold_encoded['age_group'].map({
    'Age Group1': 1, 'Age Group2': 2, 'Age Group3': 3,
    'Age Group4': 4, 'Age Group5': 5, 'Age Group6': 6
})
gold_encoded['income_level'] = gold_encoded['income_level'].str.replace(
    'Level', '', regex=False).astype(int)
gold_encoded['household_size'] = gold_encoded['household_size'].map({
    '1': 1, '2': 2, '3': 3, '4': 4, '5+': 5
})
gold_encoded['kid_count'] = gold_encoded['kid_count'].map({
    'None/Unknown': 0, '1': 1, '2': 2, '3+': 3
})
gold_encoded['home_ownership'] = gold_encoded['home_ownership'].map({
    'Renter': 0, 'Probable Renter': 1, 'Unknown': 2,
    'Probable Owner': 3, 'Homeowner': 4
})
gold_encoded['marital_status'] = gold_encoded['marital_status'].map({
    'X': 0, 'Y': 1, 'Z': 2
})
gold_encoded['log_outcome'] = np.log1p(
    gold_encoded['avg_daily_campaign_spend']
)

w_cols = [
    'pre_avg_weekly_spend', 'pre_avg_weekly_trips',
    'pre_coupon_usage_rate', 'pre_loyalty_card_rate',
    'pre_department_breadth', 'pre_active_weeks',
    'clean_window_days'
]
x_cols = [
    'age_group', 'marital_status', 'income_level',
    'household_size', 'home_ownership', 'kid_count'
]

all_covariates = w_cols + x_cols

print(f"Shape: {gold_encoded.shape}")
print(f"Treatment balance — TypeA: {gold_encoded['treatment'].sum()}, "
      f"TypeBC: {(1-gold_encoded['treatment']).sum()}")

In [0]:
# Encode the edges confirmed by PC algorithm + domain knowledge

causal_graph = """
digraph {
    treatment -> log_outcome;
    pre_avg_weekly_spend -> log_outcome;
    pre_loyalty_card_rate -> log_outcome;
    pre_loyalty_card_rate -> pre_avg_weekly_spend;
    pre_avg_weekly_spend -> pre_department_breadth;
    income_level -> pre_avg_weekly_spend;
    pre_loyalty_card_rate -> pre_avg_weekly_trips;
    pre_department_breadth -> pre_avg_weekly_trips;
    pre_active_weeks -> pre_avg_weekly_trips;
    pre_coupon_usage_rate -> pre_loyalty_card_rate;
    pre_coupon_usage_rate -> clean_window_days;
    age_group -> pre_loyalty_card_rate;
    pre_active_weeks -> pre_department_breadth;
    home_ownership -> age_group;
    kid_count -> age_group;
    marital_status -> household_size;
    home_ownership -> household_size;
    kid_count -> household_size;

    pre_avg_weekly_spend -> treatment;
    pre_avg_weekly_trips -> treatment;
    pre_coupon_usage_rate -> treatment;
    pre_loyalty_card_rate -> treatment;
    pre_department_breadth -> treatment;
    pre_active_weeks -> treatment;
    clean_window_days -> treatment;
    income_level -> treatment;
    age_group -> treatment;
}
"""

model = CausalModel(
    data=gold_encoded[['treatment', 'log_outcome'] + all_covariates],
    treatment='treatment',
    outcome='log_outcome',
    graph=causal_graph
)

# Identify the estimand
identified_estimand = model.identify_effect(
    proceed_when_unidentifiable=True
)
print(identified_estimand)

In [0]:
# Use linear regression estimator for refutation
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True,
    test_significance=True
)

print(f"DoWhy ATE estimate: {estimate.value:.4f}")
print(estimate)

## Cross-Estimator Consistency Check

Before running formal refutation tests, the DoWhy linear regression 
estimate provides a consistency check against the EconML 
CausalForestDML estimate.

| Estimator | ATE | 95% CI | p-value |
|---|---|---|---|
| CausalForestDML | -0.157 | [-0.473, 0.159] | 0.33 |
| Linear regression | -0.038 | [-0.125, 0.049] | 0.39 |

Both estimators agree on direction (negative) and conclusion 
(effect not statistically distinguishable from zero). The 
difference in magnitude reflects the parametric vs nonparametric 
nature of the two estimators: linear regression assumes constant 
effects while CausalForestDML captures nonlinear confounding. 
This consistency across estimators increases confidence in the 
finding. Formal refutation tests follow using the DoWhy estimate 
as the base.

In [0]:
# Replace real treatment with random assignment
# If the estimate changes significantly → estimate is treatment-specific (good)
# If the estimate stays the same → estimate may be spurious (bad)

with mlflow.start_run(run_name="refutation_placebo_treatment"):
    refute_placebo = model.refute_estimate(
        identified_estimand,
        estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
        num_simulations=100
    )

    print(refute_placebo)

    mlflow.log_metrics({
        "original_effect":        float(estimate.value),
        "placebo_effect":         float(refute_placebo.new_effect),
        "placebo_pvalue":         float(refute_placebo.refutation_result['p_value']),
    })

mlflow.end_run()

In [0]:
with mlflow.start_run(run_name="refutation_data_subset"):

    refute_subset = model.refute_estimate(
        identified_estimand,
        estimate,
        method_name="data_subset_refuter",
        subset_fraction=0.8,
        num_simulations=100
    )

    print(refute_subset)

    mlflow.log_metrics({
        "original_effect":    float(estimate.value),
        "subset_effect":      float(refute_subset.new_effect),
        "subset_pvalue":      float(refute_subset.refutation_result['p_value']),
    })

mlflow.end_run()

In [0]:
# Add a random variable as a confounder
# The estimate should not change much — if it does, the model is sensitive to unobserved confounding

with mlflow.start_run(run_name="refutation_random_cause"):

    refute_random = model.refute_estimate(
        identified_estimand,
        estimate,
        method_name="random_common_cause",
        num_simulations=100
    )

    print(refute_random)

    mlflow.log_metrics({
        "original_effect":      float(estimate.value),
        "random_cause_effect":  float(refute_random.new_effect),
        "random_cause_pvalue":  float(refute_random.refutation_result['p_value']),
    })

mlflow.end_run()

## Refutation Results

Three refutation tests were run on the DoWhy linear regression 
estimate (ATE = -0.038) to assess robustness of the causal finding.

| Test | Original ATE | New ATE | p-value | Result |
|---|---|---|---|---|
| Placebo treatment | -0.038 | +0.005 | 0.82 | PASS |
| Data subset (80%) | -0.038 | -0.040 | 0.84 | PASS |
| Random common cause | -0.038 | -0.038 | 0.98 | PASS |

### Interpreting Each Test

**Placebo treatment (p = 0.82)**
When real treatment labels are replaced with randomly permuted 
assignments, the estimated effect collapses to +0.005 — 
statistically indistinguishable from zero (p = 0.82). This 
confirms the original estimate of -0.038 is specific to the 
actual treatment variable and not a statistical artifact of 
the model structure. PASS.

**Data subset (p = 0.84)**
Estimating the effect on a random 80% subsample produces an 
ATE of -0.040, nearly identical to the full-sample estimate 
of -0.038 (p = 0.84 — no significant difference). The estimate 
is stable across random subsets, meaning it is not driven by 
a small number of influential observations. PASS.

**Random common cause (p = 0.98)**
Adding a randomly generated variable as an additional confounder 
produces an ATE of -0.038, unchanged from the original (p = 0.98). 
The estimate is insensitive to the addition of noise variables, 
suggesting the model is not overfitting to spurious correlations 
in the covariate space. PASS.

### Conclusion

All three refutation tests pass. The causal estimate is:
- Treatment-specific (not a modeling artifact)
- Stable across data subsets (not driven by outliers)
- Robust to spurious confounders (not overfitting)

The finding that TypeA personalized campaigns do not produce 
statistically significant spend lift over TypeBC non-personalized 
campaigns is robust to these sensitivity checks. The result holds 
under both the nonparametric CausalForestDML estimator (ATE = -0.157, 
95% CI [-0.473, 0.159]) and the linear regression estimator 
(ATE = -0.038, 95% CI [-0.125, 0.049]), with consistent refutation 
test results confirming the null finding is not a statistical artifact.
